In [1]:
import yfinance as yf

datos_extendido = yf.download("GC=F", period="10y", interval="1d")
datos_extendido.head()


[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,GC=F,GC=F,GC=F,GC=F,GC=F
Date,,,,,
2016-08-10,1344.300049,1354.199951,1344.300049,1344.800049,110
2016-08-11,1342.500000,1350.400024,1334.800049,1341.599976,666
2016-08-12,1335.800049,1351.800049,1333.199951,1336.800049,113
2016-08-15,1340.300049,1340.599976,1333.599976,1336.500000,62
2016-08-16,1350.500000,1355.000000,1341.000000,1342.199951,338


In [2]:
datos_extendido.columns = datos_extendido.columns.get_level_values(0)
datos_extendido.columns.name = None

datos_extendido = datos_extendido[["Open", "High", "Low", "Close", "Volume"]]

datos_extendido.head()

,Open,High,Low,Close,Volume
Date,,,,,
2016-08-10,1344.800049,1354.199951,1344.300049,1344.300049,110
2016-08-11,1341.599976,1350.400024,1334.800049,1342.500000,666
2016-08-12,1336.800049,1351.800049,1333.199951,1335.800049,113
2016-08-15,1336.500000,1340.599976,1333.599976,1340.300049,62
2016-08-16,1342.199951,1355.000000,1341.000000,1350.500000,338


In [3]:
datos_extendido.to_csv("../data/XAUUSD_10y.csv")
print("Datos de 10 años guardados correctamente")

Datos de 10 años guardados correctamente


In [4]:
import sys
sys.path.append("..")

from src.strategies.cruce_medias import cruce_medias

datos_extendido = cruce_medias(datos_extendido)

datos_extendido[["Close", "MA_rapida", "MA_lenta", "Señal"]].tail()

,Close,MA_rapida,MA_lenta,Señal
Date,,,,
2026-08-04,4095.399902,4058.615027,4182.082012,-1
2026-08-05,4245.799805,4067.360022,4176.578008,-1
2026-08-06,4242.000000,4072.930017,4171.410010,-1
2026-08-07,4340.700195,4084.760022,4169.274014,-1
2026-08-10,4391.899902,4104.505017,4167.126016,-1


In [5]:
datos_extendido["Cambio"] = datos_extendido["Señal"].diff()
datos_extendido["Tramo"] = (datos_extendido["Cambio"] != 0).cumsum()

tabla_extendida = datos_extendido.reset_index()

resumen_tramos_extendido = tabla_extendida.groupby("Tramo").agg(
    Fecha_inicio=("Date", "min"),
    Fecha_fin=("Date", "max"),
    Dias=("Date", "count"),
    Señal=("Señal", "first"),
    Retorno_total=("Retorno_estrategia", lambda x: (1 + x).prod() - 1)
)

print(f"Total de tramos (incluyendo calentamiento): {len(resumen_tramos_extendido)}")
resumen_tramos_extendido.tail(10)

Total de tramos (incluyendo calentamiento): 50


,Fecha_inicio,Fecha_fin,Dias,Señal,Retorno_total
Tramo,,,,,
41,2023-10-30,2024-01-30,63,1,0.013794
42,2024-01-31,2024-03-04,23,-1,-0.025862
43,2024-03-05,2024-06-14,72,1,0.084605
44,2024-06-17,2024-07-12,18,-1,-0.051427
45,2024-07-15,2024-11-25,95,1,0.076046
46,2024-11-26,2025-01-15,34,-1,-0.035240
47,2025-01-16,2025-08-19,149,1,0.191374
48,2025-08-20,2025-08-22,3,-1,-0.000346
49,2025-08-25,2026-03-25,147,1,0.348808


In [6]:
from src.backtesting.metricas import calcular_metricas

metricas_extendidas = calcular_metricas(datos_extendido, resumen_tramos_extendido)

for nombre, valor in metricas_extendidas.items():
    print(f"{nombre}: {valor}")

retorno_total: 0.3752315558158812
drawdown_maximo: -0.3427819442591825
sharpe_ratio: 0.2756318156820349
win_rate: 0.3877551020408163
total_operaciones: 49


In [8]:
from scipy import stats

retornos = datos_extendido["Retorno_estrategia"].dropna()

t_stat, p_value = stats.ttest_1samp(retornos, 0)

print(f"Estadístico t: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

Estadístico t: 0.8701
P-value: 0.3843


In [9]:
retorno_buy_hold = (datos_extendido["Close"].iloc[-1] / datos_extendido["Close"].iloc[0]) - 1

print(f"Retorno de la estrategia (cruce de medias): {metricas_extendidas['retorno_total']:.2%}")
print(f"Retorno de buy and hold (comprar y mantener): {retorno_buy_hold:.2%}")

Retorno de la estrategia (cruce de medias): 37.52%
Retorno de buy and hold (comprar y mantener): 226.71%


In [10]:
resumen_tramos_extendido["Gano"] = resumen_tramos_extendido["Retorno_total"] > 0

resumen_tramos_extendido["Racha"] = (resumen_tramos_extendido["Gano"] != resumen_tramos_extendido["Gano"].shift()).cumsum()

rachas = resumen_tramos_extendido[resumen_tramos_extendido["Señal"] != 0].groupby("Racha").agg(
    Resultado=("Gano", "first"),
    Longitud=("Gano", "count")
)

peor_racha_perdedora = rachas[rachas["Resultado"] == False]["Longitud"].max()
print(f"Peor racha de pérdidas consecutivas observada: {peor_racha_perdedora} operaciones")

Peor racha de pérdidas consecutivas observada: 6 operaciones


In [11]:
import numpy as np

np.random.seed(42)

retornos_por_operacion = resumen_tramos_extendido[resumen_tramos_extendido["Señal"] != 0]["Retorno_total"].values

n_simulaciones = 10000
peores_rachas = []

for i in range(n_simulaciones):
    secuencia_simulada = np.random.choice(retornos_por_operacion, size=len(retornos_por_operacion), replace=True)
    gano_simulado = secuencia_simulada > 0

    racha_actual = 0
    peor_racha = 0
    for resultado in gano_simulado:
        if resultado == False:
            racha_actual += 1
            peor_racha = max(peor_racha, racha_actual)
        else:
            racha_actual = 0

    peores_rachas.append(peor_racha)

peores_rachas = np.array(peores_rachas)

print(f"Peor racha promedio (en 10,000 simulaciones): {peores_rachas.mean():.1f}")
print(f"Peor racha máxima observada en simulaciones: {peores_rachas.max()}")
print(f"Percentil 95 (racha que solo se supera el 5% de las veces): {np.percentile(peores_rachas, 95):.0f}")

Peor racha promedio (en 10,000 simulaciones): 6.7
Peor racha máxima observada en simulaciones: 24
Percentil 95 (racha que solo se supera el 5% de las veces): 11


In [12]:
def capital_restante(riesgo_por_operacion, num_perdidas_seguidas):
    return (1 - riesgo_por_operacion) ** num_perdidas_seguidas

escenarios_riesgo = [0.01, 0.02, 0.03, 0.05, 0.10]
rachas_a_probar = [6, 11, 24]

for riesgo in escenarios_riesgo:
    print(f"\nSi arriesgas {riesgo:.0%} de tu capital por operación:")
    for racha in rachas_a_probar:
        restante = capital_restante(riesgo, racha)
        print(f"  Tras {racha} pérdidas seguidas -> te queda el {restante:.1%} del capital original")


Si arriesgas 1% de tu capital por operación:
  Tras 6 pérdidas seguidas -> te queda el 94.1% del capital original
  Tras 11 pérdidas seguidas -> te queda el 89.5% del capital original
  Tras 24 pérdidas seguidas -> te queda el 78.6% del capital original

Si arriesgas 2% de tu capital por operación:
  Tras 6 pérdidas seguidas -> te queda el 88.6% del capital original
  Tras 11 pérdidas seguidas -> te queda el 80.1% del capital original
  Tras 24 pérdidas seguidas -> te queda el 61.6% del capital original

Si arriesgas 3% de tu capital por operación:
  Tras 6 pérdidas seguidas -> te queda el 83.3% del capital original
  Tras 11 pérdidas seguidas -> te queda el 71.5% del capital original
  Tras 24 pérdidas seguidas -> te queda el 48.1% del capital original

Si arriesgas 5% de tu capital por operación:
  Tras 6 pérdidas seguidas -> te queda el 73.5% del capital original
  Tras 11 pérdidas seguidas -> te queda el 56.9% del capital original
  Tras 24 pérdidas seguidas -> te queda el 29.2% d

In [13]:
retornos_diarios = datos_extendido["Retorno_estrategia"].dropna()

VaR_95 = np.percentile(retornos_diarios, 5)
volatilidad_historica = retornos_diarios.std()

print(f"VaR 95% (diario): {VaR_95:.2%}")
print(f"Volatilidad histórica (diaria): {volatilidad_historica:.2%}")
print(f"Volatilidad histórica (anualizada): {volatilidad_historica * (252 ** 0.5):.2%}")

VaR 95% (diario): -1.61%
Volatilidad histórica (diaria): 1.05%
Volatilidad histórica (anualizada): 16.66%
